split into CVs

add correlation-based selection

1) load and analyze variance of descriptors in all the datasets
2) remove descriptors with low variance
4) add gaussian processes regression and bayesian linear regression 
5) add hyperparameter optimization

when done with all the datasets, analyze which descriptors are most important for each property
select ~20

debug round 5

- optimize parallel script in terms of logging and run on structures

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from pathlib import Path
import scipy.stats as stats
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import glob
import os
import itertools
pd.set_option('display.max_rows', None)
warnings.filterwarnings('ignore')
from collections import Counter
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr

from utils.load_results_to_dataframe import load_json_results

## Load data

Experimental table (`data/pdgf38.csv`), Propermab features (`pdgf38_propermab/features.csv`), and developability JSON descriptors (`pdgf38_results/`). For Propermab, `name` is taken from the stem of `pdb_file` (e.g. `AB-001` from `.../AB-001.pdb`). Each feature table is merged to the experimental rows on `name` and sorted by `name`.

In [ ]:
def _repo_root() -> Path:
    """Resolve repo root whether the notebook cwd is the repo or ``src/``."""
    here = Path.cwd().resolve()
    for base in (here, here.parent):
        if (base / "data" / "pdgf38.csv").is_file():
            return base
    raise FileNotFoundError("Could not locate data/pdgf38.csv; run the notebook from the repo root or from src/.")


REPO = _repo_root()
NAME_COL = "name"

exp_path = REPO / "data" / "pdgf38.csv"
propermab_path = REPO / "pdgf38_propermab" / "features.csv"
ours_dir = REPO / "pdgf38_results"

exp = pd.read_csv(exp_path)
propermab = pd.read_csv(propermab_path)
ours = load_json_results(ours_dir)

# Propermab export: IDs live in pdb_file paths (e.g. .../pdgf38/AB-001.pdb -> AB-001)
if "pdb_file" in propermab.columns:
    propermab[NAME_COL] = propermab["pdb_file"].astype(str).map(lambda p: Path(p).stem)
elif NAME_COL not in propermab.columns:
    raise ValueError("propermab features must have pdb_file or name for merging")

for df in (exp, propermab, ours):
    if NAME_COL not in df.columns:
        raise ValueError(f"Missing {NAME_COL!r} column in {df.shape}")

# Align key dtype for stable merges (handles int vs str IDs)
exp[NAME_COL] = exp[NAME_COL].astype(str)
propermab[NAME_COL] = propermab[NAME_COL].astype(str)
ours[NAME_COL] = ours[NAME_COL].astype(str)

merged_propermab = exp.merge(propermab, on=NAME_COL, how="inner", suffixes=("", "_propermab"))
merged_ours = exp.merge(ours, on=NAME_COL, how="inner", suffixes=("", "_ours"))

merged_propermab = merged_propermab.sort_values(NAME_COL).reset_index(drop=True)
merged_ours = merged_ours.sort_values(NAME_COL).reset_index(drop=True)

print(f"exp: {len(exp)} rows; Propermab merge: {len(merged_propermab)} rows; our JSON merge: {len(merged_ours)} rows")

# Utils

In [9]:
def load_and_merge(exp_csv_path: Path, results_dir_path: Path, base: str):
    """Load experimental CSV + descriptor JSONs, then inner-join on `name`."""
    df_results = load_json_results(results_dir_path)
    df_results["base"] = base

    exp_df = pd.read_csv(exp_csv_path)

    if "name" not in exp_df.columns:
        raise ValueError(f"Experimental CSV {exp_csv_path} must contain a 'name' column.")
    if "name" not in df_results.columns:
        raise ValueError(
            f"Results loaded from {results_dir_path} are missing a 'name' column for merging."
        )

    try:
        exp_df["name"] = exp_df["name"].astype(int)
    except:
        pass
    try:
        df_results["name"] = df_results["name"].astype(int)
    except:
        pass

    exp_df["name"] = exp_df["name"].astype(str)
    df_results["name"] = df_results["name"].astype(str)

    merged_df = exp_df.merge(df_results, on="name", how="inner")

    # One-hot-encode feature_* columns only when values contain alphabetic tokens.
    oh_from = []
    for col in [c for c in list(merged_df.columns) if str(c).startswith("feature_")]:
        s = merged_df[col]
        non_missing = s[s.notna()]
        if non_missing.empty:
            continue

        non_missing_str = non_missing.astype(str).str.strip()
        if not non_missing_str.str.contains(r"[A-Za-z]", regex=True).any():
            continue

        dummies = pd.get_dummies(
            s, prefix=str(col), prefix_sep="__", dtype=float, dummy_na=True
        )
        merged_df = merged_df.drop(columns=[col])
        merged_df = pd.concat([merged_df, dummies], axis=1)
        oh_from.append(col)

    print(
        f"Merged experimental {Path(exp_csv_path).name} with results '{base}': {len(merged_df)} rows"
    )
    if oh_from:
        print(
            "One-hot-encoded feature columns (alphabetic/alphanumeric values): "
            + ", ".join(oh_from)
        )
    return merged_df


In [10]:
from automl.feature_selectors import remove_low_variance_features


In [11]:
def _bh_adjust(pvals):
    """Benjamini-Hochberg FDR adjustment. pvals: array-like. NaNs are preserved (not used in adjustment)."""
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    valid = np.isfinite(p)
    if not np.any(valid):
        return p
    p_valid = p[valid]
    n = len(p_valid)
    order = np.argsort(p_valid)
    p_sorted = p_valid[order]
    ratios = n * p_sorted / np.arange(1, n + 1)
    adj_sorted = np.minimum(1, np.minimum.accumulate(ratios[::-1])[::-1])
    rank_of_original = np.argsort(order)
    out[valid] = adj_sorted[rank_of_original]
    return out

def calculate_correlations_and_plot(
    merged_df,
    target_col,
    p_threshold=0.05,
    fdr_alpha=0.05,
    use_fdr=True,
    normalize=False,
    make_plots=False,
    correlation_threshold=None, 
):
    """Compute Spearman correlations for one or multiple targets.

    Parameters
    ----------
    target_col : str or list[str]
        Target column name(s).

    Returns
    -------
    corr_df_by_target : pd.DataFrame or dict
        For a single target: the correlation dataframe.
        For multiple targets: {target_col: corr_df}.

    significant_by_target : dict
        {target_col: [(feature, spearman_r), ...]}
    """

    if isinstance(target_col, str):
        target_cols = [target_col]
        scalar_target = True
    else:
        # Accept list/tuple/set/np.ndarray/pd.Series.
        target_cols = list(target_col)
        scalar_target = False

    if len(target_cols) == 0:
        raise ValueError("target_col must be a non-empty string or iterable of strings")

    exclude_cols = ['antibody_id', 'residue_number', 'n_total_rows', 'n_filtered_rows', 'n_beta_sheet_rows', 'n_exposed_rows']
    id_cols = {'structure_id', 'base', 'heavy', 'light', 'dataset', 'name', 'antibody_name'}

    corr_df_by_target = {}
    significant_by_target = {}

    for tcol in target_cols:
        merged_df_t = merged_df.copy()
        merged_df_t[tcol] = pd.to_numeric(merged_df_t[tcol], errors="coerce")

        n_before = len(merged_df_t)
        merged_df_t = merged_df_t.dropna(subset=[tcol])
        n_after = len(merged_df_t)
        if n_before > n_after:
            print(f"Dropped {n_before - n_after} rows with NaN/invalid target '{tcol}' (using {n_after} for correlations).")

        numeric_cols = merged_df_t.select_dtypes(include=[np.number]).columns.tolist()
        if len(numeric_cols) > 0:
            feature_cols = [
                col
                for col in numeric_cols
                if col not in exclude_cols
                and col not in id_cols
                and col not in target_cols
                and not str(col).startswith("target")
            ]
        else:
            feature_cols = [
                col
                for col in merged_df_t.columns
                if col not in exclude_cols
                and col not in id_cols
                and col not in target_cols
                and not str(col).startswith("target")
            ]

        correlations = []
        for col in feature_cols:
            data = merged_df_t[[tcol, col]].copy()
            data[tcol] = pd.to_numeric(data[tcol], errors='coerce')
            data[col] = pd.to_numeric(data[col], errors='coerce')
            data = data.dropna()
            if len(data) < 3:
                continue

            if normalize:
                target_min = data[tcol].min()
                target_max = data[tcol].max()
                target_range = target_max - target_min
                if target_range > 0:
                    target_norm = (data[tcol] - target_min) / target_range
                else:
                    target_norm = data[tcol]

                feature_min = data[col].min()
                feature_max = data[col].max()
                feature_range = feature_max - feature_min
                if feature_range > 0:
                    feature_norm = (data[col] - feature_min) / feature_range
                else:
                    feature_norm = data[col]

                spearman_r, spearman_p = spearmanr(target_norm, feature_norm)
            else:
                spearman_r, spearman_p = spearmanr(data[tcol], data[col])

            correlations.append({
                'feature': col,
                'spearman_r': spearman_r,
                'spearman_p': spearman_p,
                'n_samples': len(data)
            })

        corr_df = pd.DataFrame(correlations)
        if corr_df.empty:
            corr_df = pd.DataFrame(columns=['feature', 'spearman_r', 'spearman_p', 'spearman_p_adj', 'n_samples'])
            significant = corr_df.copy()
        else:
            if use_fdr:
                corr_df['spearman_p_adj'] = _bh_adjust(corr_df['spearman_p'].values)
                significant = corr_df[corr_df['spearman_p_adj'] < fdr_alpha].copy()
            else:
                corr_df['spearman_p_adj'] = corr_df['spearman_p'].values
                significant = corr_df[corr_df['spearman_p'] < p_threshold].copy()

            # NEW: apply correlation magnitude threshold
            if correlation_threshold is not None:
                significant = significant[
                    significant["spearman_r"].abs() >= correlation_threshold
                ].copy()

        print(f"[target={tcol}] Total features tested: {len(corr_df)}")
        print(f"[target={tcol}] Significant (raw p < {p_threshold}): {(corr_df['spearman_p'] < p_threshold).sum()}")
        if use_fdr:
            print(f"[target={tcol}] Significant after FDR correction (adj p < {fdr_alpha}): {len(significant)}")
        else:
            print(f"[target={tcol}] Significant (no FDR): {len(significant)}")
        if correlation_threshold is not None:
            print(
                f"[target={tcol}] After applying |rho| >= {correlation_threshold}: {len(significant)}"
            )
        if normalize:
            print(f"[target={tcol}] Note: Features were min-max normalized before correlation calculation")
        print(
            f"[target={tcol}] Top correlations by absolute Spearman r ({'FDR-significant only' if use_fdr else 'raw p < ' + str(p_threshold)}):"
        )

        if len(significant) > 0:
            sig = significant.copy()
            sig["spearman_r"] = pd.to_numeric(sig["spearman_r"], errors="coerce")
            sig = sig.dropna(subset=["spearman_r"])
            if len(sig) > 0:
                cols = ["feature", "spearman_r", "spearman_p", "spearman_p_adj"]
                print(sig.nlargest(10, "spearman_r", keep="all")[[c for c in cols if c in sig.columns]])
            else:
                print("(none)")
        else:
            print("(no significant correlations)")

        if make_plots and len(significant) > 0:
            n_plots = len(significant)
            n_cols = 3
            n_rows = (n_plots + n_cols - 1) // n_cols

            fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
            axes = np.atleast_1d(axes).flatten()

            for plot_idx, (idx, row) in enumerate(significant.iterrows()):
                ax = axes[plot_idx]
                data = merged_df_t[[tcol, row['feature']]].copy()
                data[tcol] = pd.to_numeric(data[tcol], errors='coerce')
                data[row['feature']] = pd.to_numeric(data[row['feature']], errors='coerce')
                data = data.dropna()

                if normalize:
                    target_min = data[tcol].min()
                    target_max = data[tcol].max()
                    target_range = target_max - target_min
                    if target_range > 0:
                        target_norm = (data[tcol] - target_min) / target_range
                    else:
                        target_norm = data[tcol]

                    feature_min = data[row['feature']].min()
                    feature_max = data[row['feature']].max()
                    feature_range = feature_max - feature_min
                    if feature_range > 0:
                        feature_norm = (data[row['feature']] - feature_min) / feature_range
                    else:
                        feature_norm = data[row['feature']]

                    ax.scatter(feature_norm, target_norm, alpha=0.6)

                    z = np.polyfit(feature_norm, target_norm, 1)
                    p = np.poly1d(z)
                    # ax.plot(feature_norm, p(feature_norm), "r--", alpha=0.8)
                else:
                    ax.scatter(data[row['feature']], data[tcol], alpha=0.6)

                    z = np.polyfit(data[row['feature']], data[tcol], 1)
                    p = np.poly1d(z)
                    # ax.plot(data[row['feature']], p(data[row['feature']]), "r--", alpha=0.8)

                ax.set_xlabel(row['feature'], fontsize=10)
                ax.set_ylabel(tcol, fontsize=10)
                p_label = 'p_adj' if use_fdr else 'p'
                ax.set_title(f"ρ={row['spearman_r']:.3f}, {p_label}={row['spearman_p_adj']:.3e}", fontsize=9)
                ax.grid(True, alpha=0.3)

            for idx in range(len(significant), len(axes)):
                axes[idx].axis('off')

            plt.tight_layout()
            plt.show()

        # Return list of (feature, spearman_r) tuples sorted by |spearman_r|.
        significant_tuples = []
        if len(significant) > 0:
            sig = significant.copy()
            sig["spearman_r"] = pd.to_numeric(sig["spearman_r"], errors="coerce")
            sig = sig.dropna(subset=["spearman_r"])
            if len(sig) > 0:
                sig = sig.assign(_abs_r=sig["spearman_r"].abs())
                sig = sig.sort_values("_abs_r", ascending=False).drop(columns=["_abs_r"])
                significant_tuples = list(
                    zip(
                        sig["feature"].astype(str).tolist(),
                        sig["spearman_r"].astype(float).tolist(),
                    )
                )

        corr_df_by_target[tcol] = corr_df
        significant_by_target[tcol] = significant_tuples

    if scalar_target:
        return corr_df_by_target[target_cols[0]], significant_by_target

    return corr_df_by_target, significant_by_target

In [12]:
from automl.feature_selectors import stability_selection_features




In [13]:
from sklearn.linear_model import ElasticNet, Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from scipy.stats import spearmanr, pearsonr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def fit_and_compare_models(
    merged_train,
    merged_test,
    target_col,
    feature_list,
    enet_alpha,              # alpha chosen outside (e.g. via CV)
    random_state=42,
    enet_l1_ratio=0.5,
    max_feature_fraction=0.2,
    make_plots=False,
):
    """
    Fit ElasticNet, Ridge, LinearRegression, and RandomForest on TRAIN data,
    evaluate on TEST data.

    - ElasticNet with provided enet_alpha is used on train set to select features:
      max number of features = n_train_samples * max_feature_fraction (top by |coef|).
    - All models are then fit on the selected feature set using train data.
    - Metrics and plots are computed on the test set.

    Returns
    -------
    selected_features : list
        Features kept after ElasticNet-based selection (from train data).
    results_df : pd.DataFrame
        Per-model metrics on the test data:
        ['model', 'r2', 'pearson_r', 'spearman_r', 'spearman_p', 'n_features', 'n_train', 'n_test'].
    features_by_model : dict[str, list[str]]
        Feature names with non-negligible weight after fitting each model (linear models:
        |coef| > 1e-8; RandomForest: importance > max(1e-10, 1e-3 * max importance)).
        SVR (RBF) and kNN list all ``selected_features`` (they do not expose sparse weights).
    """
    # Ensure we don't accidentally include the target itself as a feature (no leakage)
    feature_list = [f for f in feature_list if f not in ("target", target_col, "target_viscosity", "viscosity") and not str(f).startswith("target")]

    X_train = merged_train[feature_list].copy()
    X_test = merged_test[feature_list].copy()



    for c in feature_list:
        s_tr = X_train[c]
        s_te = X_test[c]

        # Numeric-only preprocessing: coerce all feature values to numeric.
        # Non-numeric values become NaN and are filtered by train/test masks below.
        X_train[c] = pd.to_numeric(s_tr, errors="coerce")
        X_test[c] = pd.to_numeric(s_te, errors="coerce")

    y_train = pd.to_numeric(merged_train[target_col], errors="coerce")
    y_test = pd.to_numeric(merged_test[target_col], errors="coerce")

    train_mask = y_train.notna() & X_train.notna().all(axis=1)
    test_mask = y_test.notna() & X_test.notna().all(axis=1)

    X_train = X_train.loc[train_mask].values
    y_train = y_train.loc[train_mask].values
    X_test = X_test.loc[test_mask].values
    y_test = y_test.loc[test_mask].values

    n_train = len(y_train)
    n_test = len(y_test)

    if n_train < 3 or n_test < 1:
        raise ValueError("Not enough samples to fit/evaluate models.")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ElasticNet-based feature selection using externally provided alpha (train only)
    enet = ElasticNet(
        alpha=enet_alpha,
        l1_ratio=enet_l1_ratio,
        max_iter=20000,
        tol=1e-3,
        random_state=random_state,
    )
    enet.fit(X_train_scaled, y_train)

    n_keep = max(1, int(n_train * max_feature_fraction))
    n_keep = min(n_keep, len(feature_list))
    order = np.argsort(np.abs(enet.coef_))[::-1]
    top_indices = order[:n_keep]
    selected_features = [feature_list[i] for i in top_indices]

    X_train_sel = X_train_scaled[:, top_indices]
    X_test_sel = X_test_scaled[:, top_indices]

    knn_neighbors = max(1, min(5, n_train - 1))

    # Models: fit on train, evaluate on test (inputs are already scaled)
    models = {
        "ElasticNet": ElasticNet(
            alpha=enet_alpha,
            l1_ratio=enet_l1_ratio,
            max_iter=20000,
            tol=1e-3,
            random_state=random_state,
        ),
        "Ridge": Ridge(alpha=enet_alpha, random_state=random_state),
        "Linear": LinearRegression(),
        "RandomForest": RandomForestRegressor(
            n_estimators=100,
            max_features="sqrt",
            random_state=random_state,
        ),
        "SVR": SVR(kernel="rbf", C=1.0, epsilon=0.1, gamma="scale"),
        "kNN": KNeighborsRegressor(
            n_neighbors=knn_neighbors,
            weights="distance",
        ),
    }

    results = []
    preds = {}  # for plotting
    for name, model in models.items():
        model.fit(X_train_sel, y_train)
        y_pred = model.predict(X_test_sel)
        preds[name] = y_pred
        r2 = r2_score(y_test, y_pred)
        pr_r, pr_p = pearsonr(y_test, y_pred)
        sp_r, sp_p = spearmanr(y_test, y_pred)
        results.append(
            {
                "model": name,
                "r2": float(r2) if not np.isnan(r2) else np.nan,
                "pearson_r": float(pr_r) if not np.isnan(pr_r) else np.nan,
                "spearman_r": float(sp_r) if not np.isnan(sp_r) else np.nan,
                "spearman_p": float(sp_p) if not np.isnan(sp_p) else np.nan,
                "n_features": len(selected_features),
                "n_train": n_train,
                "n_test": n_test,
            }
        )

    results_df = pd.DataFrame(results)

    # Per-model feature names actually carrying weight after refit (subset of selected_features).
    _COEF_TOL = 1e-8
    _IMP_TOL = 1e-10

    def _active_linear(coef) -> list[str]:
        v = np.asarray(coef).ravel()
        if v.size != len(selected_features):
            return list(selected_features)
        mask = np.abs(v) > _COEF_TOL
        return [selected_features[i] for i in np.where(mask)[0]]

    features_by_model: dict[str, list[str]] = {}
    for name, model in models.items():
        if name in ("ElasticNet", "Ridge", "Linear"):
            features_by_model[name] = _active_linear(model.coef_)
        elif name == "RandomForest":
            imp = np.asarray(model.feature_importances_, dtype=float)
            if imp.size != len(selected_features):
                features_by_model[name] = list(selected_features)
            else:
                imp_max = float(np.max(imp)) if imp.size else 0.0
                thr = max(_IMP_TOL, 1e-3 * imp_max)
                mask = imp > thr
                features_by_model[name] = [
                    selected_features[i] for i in np.where(mask)[0]
                ]
        else:
            # RBF SVR and kNN use all inputs; no sparse linear structure to report.
            features_by_model[name] = list(selected_features)

    # --- Plots (on test set) ---
    # 1) Observed vs predicted (one panel per model)
    if make_plots:
        n_models = len(preds)
        fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4))
        axes = np.atleast_1d(axes)
        for ax, (name, y_pred) in zip(axes, preds.items()):
            ax.scatter(y_test, y_pred, alpha=0.7, edgecolors="k", linewidths=0.5)
            lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
            ax.plot(lims, lims, "r--", label="y = ŷ")
            ax.set_xlabel("Observed (test)")
            ax.set_ylabel("Predicted (test)")
            ax.set_title(f"{name} (train→test, n_test={n_test})")
            ax.legend(loc="upper left", fontsize=8)
            ax.set_aspect("equal", adjustable="box")
            ax.grid(True, alpha=0.3)
        plt.suptitle(
            f"Test predictions (train fit, no internal CV)\nX_test shape: ({n_test}, {len(selected_features)})",
            fontsize=11,
        )
        plt.tight_layout()
        plt.show()

        # 2) Bar chart: R² and Spearman r per model (on test)
        fig, ax = plt.subplots(figsize=(6, 4))
        x = np.arange(len(results_df))
        w = 0.35
        ax.bar(x - w / 2, results_df["r2"], w, label="R²", color="steelblue", alpha=0.8)
        ax.bar(x + w / 2, results_df["spearman_r"], w, label="Spearman r", color="coral", alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(results_df["model"])
        ax.set_ylabel("Score (test)")
        ax.legend()
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    return selected_features, results_df, features_by_model

In [14]:
from sklearn.feature_selection import mutual_info_regression

def reduce_correlated_features(
    split_train_df: "pd.DataFrame",
    target_cols=None,
    feature_set=None,
    correlation_threshold: float = 0.85,
    importance_metric: str = "spearman",
):
    """Reduce highly correlated features within ONE train split.

    This is done per target column, returning a dict:
    ``{target_col: kept_features}``.

    Parameters
    ----------
    split_train_df:
        Training split dataframe.
    target_cols:
        Target column name(s). If None, tries to infer from ``feature_set``
        keys (dict) or from a global ``target_cols`` variable.
    feature_set:
        Optional feature set(s) to restrict the pruning.
        - list[str]: applies to all targets
        - dict[target_col, list[str]]: per-target restriction
        - None: use all numeric columns (excluding target/ID-like columns)

    Returns
    -------
    reduced_features_by_target : dict[str, list[str]]
    """

    import pandas as pd
    import numpy as np
    from scipy.stats import spearmanr
    from scipy.cluster.hierarchy import linkage, fcluster
    from scipy.spatial.distance import squareform

    importance_metric = importance_metric.lower()
    if importance_metric != "spearman":
        raise ValueError("importance_metric must be 'spearman' for hierarchical clustering-based pruning")

    # Normalize target_cols input / inference.
    if target_cols is None:
        if isinstance(feature_set, dict) and len(feature_set) > 0:
            target_cols = list(feature_set.keys())
        elif "target_cols" in globals() and globals().get("target_cols") is not None:
            target_cols = list(globals().get("target_cols"))
        else:
            raise ValueError("target_cols must be provided (or inferred from feature_set keys).")

    if isinstance(target_cols, str):
        target_cols = [target_cols]
    else:
        target_cols = list(target_cols)

    target_cols = [str(t) for t in target_cols]
    if not target_cols:
        raise ValueError("target_cols must be non-empty")

    # Global exclude list to avoid pruning IDs/metadata and avoid selecting targets as features.
    global_exclude_cols = {
        "antibody_id",
        "structure_id",
        "residue_number",
        "n_total_rows",
        "n_filtered_rows",
        "n_beta_sheet_rows",
        "n_exposed_rows",
        "base",
        "heavy",
        "light",
        "name",
        "dataset",
        "antibody_name",
        "index",
        "target",
        "viscosity",
        "target_viscosity",
    }

    # Interpret feature_set as a per-target restriction.
    feature_set_by_target = {}
    if feature_set is None:
        feature_set_by_target = {t: None for t in target_cols}
    elif isinstance(feature_set, dict):
        for t in target_cols:
            feature_set_by_target[t] = feature_set.get(t, None)
    else:
        # Treat list/iterable as one shared feature set.
        shared = list(feature_set)
        feature_set_by_target = {t: shared for t in target_cols}

    reduced_features_by_target = {}

    for target_col in target_cols:
        # Determine candidate features for this target.
        if feature_set_by_target.get(target_col) is None:
            numeric_cols = split_train_df.select_dtypes(include=[np.number]).columns.tolist()
            sig_features = [
                f
                for f in numeric_cols
                if f not in global_exclude_cols
                and f not in target_cols
                and not str(f).startswith("target")
            ]
        else:
            fs = set(feature_set_by_target[target_col])
            sig_features = [
                f
                for f in fs
                if f in split_train_df.columns
                and f not in global_exclude_cols
                and f not in target_cols
                and not str(f).startswith("target")
            ]

        if len(sig_features) < 2:
            reduced_features_by_target[target_col] = sig_features
            continue

        # Prepare numeric data including the target (one-hot low numeric-fraction columns).
        data = split_train_df[sig_features + [target_col]].copy()
        pos = 0
        while pos < len(sig_features):
            c = sig_features[pos]
            s = data[c]
            s_num = pd.to_numeric(s, errors="coerce")

            non_missing = int(s.notna().sum())
            converted = int(s_num.notna().sum())

            if non_missing == 0:
                data[c] = s_num
                pos += 1
            elif converted >= max(1, int(0.8 * non_missing)):
                data[c] = s_num
                pos += 1
            else:
                dummies = pd.get_dummies(
                    s, prefix=str(c), prefix_sep="__", dtype=float, dummy_na=True
                )
                data = data.drop(columns=[c])
                data = pd.concat([data, dummies], axis=1)
                sig_features = (
                    sig_features[:pos] + sig_features[pos + 1 :] + list(dummies.columns)
                )
        data[target_col] = pd.to_numeric(data[target_col], errors="coerce")
        data = data.dropna(how="all")
        if len(data) < 3:
            reduced_features_by_target[target_col] = sig_features
            continue

        # Target importance per feature (to decide which member of a correlated pair to keep).
        target_score = {}
        for f in sig_features:
            sub = data[[f, target_col]].dropna()
            if len(sub) < 3:
                target_score[f] = 0.0
                continue

            r, _ = spearmanr(sub[f], sub[target_col])
            score = float(r) if not np.isnan(r) else 0.0

            target_score[f] = score

        # 1) Spearman feature-feature correlation matrix.
        corr = data[sig_features].corr(method="spearman").abs().fillna(0.0)
        np.fill_diagonal(corr.values, 1.0)

        # 2) Hierarchical clustering on distance = 1 - |corr|.
        dist = 1.0 - corr
        np.fill_diagonal(dist.values, 0.0)

        if len(sig_features) == 2:
            cluster_labels = np.array([1, 1]) if dist.iloc[0, 1] <= (1.0 - correlation_threshold) else np.array([1, 2])
        else:
            condensed = squareform(dist.values, checks=False)
            Z = linkage(condensed, method="average")
            cluster_labels = fcluster(Z, t=(1.0 - correlation_threshold), criterion="distance")

        # 3) One representative per cluster: highest |corr(feature, target)|.
        cluster_to_features = {}
        for feat, label in zip(sig_features, cluster_labels):
            cluster_to_features.setdefault(int(label), []).append(feat)

        kept_features = []
        for label in sorted(cluster_to_features):
            members = cluster_to_features[label]
            best_feat = max(members, key=lambda f: abs(target_score.get(f, 0.0)))
            kept_features.append(best_feat)

        # Keep original feature ordering for downstream reproducibility.
        kept_features = [f for f in sig_features if f in set(kept_features)]
        reduced_features_by_target[target_col] = kept_features

    return reduced_features_by_target

In [15]:
from automl.feature_selectors import (
    select_features_forward_sfs,
    select_features_floating_sfs,
    select_features_rfe,
    select_features_by_target_correlation,
)
from automl.utils import compute_correlation_bundle


# Load data

In [27]:
# Experimental datasets + matching descriptor result folders
df_ab21 = load_and_merge(
    exp_csv_path=Path("../data/ab21.csv"),
    results_dir_path=Path("../ab21_results"),
    base="ab21",
)

df_pdgf38 = load_and_merge(
    exp_csv_path=Path("../data/pdgf38.csv"),
    results_dir_path=Path("../pdgf38_results"),
    base="pdgf38",
)

df_garbinski2023_tm1 = load_and_merge(
    exp_csv_path=Path("../data/garbinski2023_tm1.csv"),
    results_dir_path=Path("../garbinski2023_results"),
    base="garbinski2023",
)

df_ginkgo = load_and_merge(
    exp_csv_path=Path("../data/ginkgo.csv"),
    results_dir_path=Path("../GINKGO_results"),
    base="GINKGO",
)

df_hutchinson2023enhancement_top200tm1_igg = load_and_merge(
    exp_csv_path=Path("../data/hutchinson2023enhancement_top200tm1_igg.csv"),
    results_dir_path=Path("../hutchinson2023enhancement_results"),
    base="hutchinson2023enhancement",
)

df_jain2017biophysical = load_and_merge(
    exp_csv_path=Path("../data/jain2017biophysical.csv"),
    results_dir_path=Path("../jain2017biophysical_results"),
    base="jain2017biophysical",
)

df_jain2023identifying = load_and_merge(
    exp_csv_path=Path("../data/jain2023identifying.csv"),
    results_dir_path=Path("../jain2023identifying_results"),
    base="jain2023identifying",
)

df_jain2024assessment = load_and_merge(
    exp_csv_path=Path("../data/jain2024assessment.csv"),
    results_dir_path=Path("../jain2024assessment_results"),
    base="jain2024assessment",
)

df_jetha2019homology_RT = load_and_merge(
    exp_csv_path=Path("../data/jetha2019homology_RT.csv"),
    results_dir_path=Path("../jetha2019homology_results"),
    base="jetha2019homology",
)

df_kraft2019herapin_relrt = load_and_merge(
    exp_csv_path=Path("../data/kraft2019herapin_relrt.csv"),
    results_dir_path=Path("../kraft2019herapin_results"),
    base="kraft2019herapin",
)


Merged experimental ab21.csv with results 'ab21': 21 rows
Merged experimental pdgf38.csv with results 'pdgf38': 38 rows
Merged experimental garbinski2023_tm1.csv with results 'garbinski2023': 86 rows
Merged experimental ginkgo.csv with results 'GINKGO': 246 rows
Ordinal-encoded feature columns containing alphabetic/alphanumeric values: feature_hc_subtype, feature_lc_subtype
Merged experimental hutchinson2023enhancement_top200tm1_igg.csv with results 'hutchinson2023enhancement': 192 rows
Merged experimental jain2017biophysical.csv with results 'jain2017biophysical': 137 rows
Merged experimental jain2023identifying.csv with results 'jain2023identifying': 115 rows
Merged experimental jain2024assessment.csv with results 'jain2024assessment': 43 rows
Merged experimental jetha2019homology_RT.csv with results 'jetha2019homology': 97 rows
Merged experimental kraft2019herapin_relrt.csv with results 'kraft2019herapin': 128 rows


# Load Propermab features (once)

Populates **`propermab_dfs`** for every available `*_propermab/features.csv`. Which studies enter the model is chosen in **Build models** via **`MODEL_DATASET_KEYS`** — change cohort there and rerun that cell + selection + fit only (no Propermab reload).


In [28]:
from pathlib import Path

# Load **once** after main descriptor merges. To change which studies you model, edit `MODEL_DATASET_KEYS` in **Build models** and rerun that cell + selection + fit only — **not** this cell.
# Same experimental CSVs as the main *Build model* list; features from matching `*_propermab/features.csv`.
_PROPERMAB_SPECS = [
    ("kraft2019herapin_relrt", Path("../data/kraft2019herapin_relrt.csv"), Path("../kraft2019herapin_propermab/features.csv")),
    ("jain2024assessment", Path("../data/jain2024assessment.csv"), Path("../jain2024assessment_propermab/features.csv")),
    ("jain2023identifying", Path("../data/jain2023identifying.csv"), Path("../jain2023identifying_propermab/features.csv")),
    ("jain2017biophysical", Path("../data/jain2017biophysical.csv"), Path("../jain2017biophysical_propermab/features.csv")),
    ("garbinski2023_tm1", Path("../data/garbinski2023_tm1.csv"), Path("../garbinski2023_propermab/features.csv")),
    ("ab21", Path("../data/ab21.csv"), Path("../ab21_propermab/features.csv")),
    ("pdgf38", Path("../data/pdgf38.csv"), Path("../pdgf38_propermab/features.csv")),
    ("ginkgo", Path("../data/ginkgo.csv"), Path("../GINKGO_propermab/features.csv")),
    ("hutchinson2023enhancement_top200tm1_igg", Path("../data/hutchinson2023enhancement_top200tm1_igg.csv"), Path("../hutchinson2023enhancement_propermab/features.csv")),
    ("jetha2019homology_RT", Path("../data/jetha2019homology_RT.csv"), Path("../jetha2019homology_propermab/features.csv")),
]


def _load_propermab_merged(exp_path: Path, features_path: Path) -> pd.DataFrame:
    feat_df = pd.read_csv(features_path)
    exp_df = pd.read_csv(exp_path)
    exp_df["name"] = exp_df["name"].astype(str)
    feat_df["name"] = feat_df["pdb_file"].str.split("/").str[-1].str.split(".").apply(lambda x: x[0])
    feat_df = feat_df.drop(columns=["pdb_file", "error"], errors="ignore")
    return feat_df.merge(exp_df, on="name", how="inner")


propermab_dfs = {}

for key, exp_p, feat_p in _PROPERMAB_SPECS:
    if not feat_p.exists():
        print(f"propermab_df_{key}: skip (missing {feat_p})")
        continue
    df = _load_propermab_merged(exp_p, feat_p)
    for col in [c for c in list(df.columns) if str(c).startswith("feature_")]:
        s = df[col]
        if pd.api.types.is_numeric_dtype(s):
            continue
        non_missing = s[s.notna()]
        if non_missing.empty:
            continue
        dummies = pd.get_dummies(
            s, prefix=str(col), prefix_sep="__", dtype=float, dummy_na=True
        )
        df = df.drop(columns=[col])
        df = pd.concat([df, dummies], axis=1)
    propermab_dfs[key] = df
    globals()[f"propermab_df_{key}"] = df
    print(f"propermab_df_{key}: {len(df)} rows <- {feat_p.parent.name} + {exp_p.name}")

# All successfully loaded studies, in `_PROPERMAB_SPECS` order.
propermab_datasets = [propermab_dfs[k].dropna() for k in propermab_dfs]


propermab_df_kraft2019herapin_relrt: 128 rows <- kraft2019herapin_propermab + kraft2019herapin_relrt.csv
propermab_df_jain2024assessment: 43 rows <- jain2024assessment_propermab + jain2024assessment.csv
propermab_df_jain2023identifying: 115 rows <- jain2023identifying_propermab + jain2023identifying.csv
propermab_df_jain2017biophysical: 137 rows <- jain2017biophysical_propermab + jain2017biophysical.csv
propermab_df_garbinski2023_tm1: 86 rows <- garbinski2023_propermab + garbinski2023_tm1.csv
propermab_df_ab21: 21 rows <- ab21_propermab + ab21.csv
propermab_df_pdgf38: 38 rows <- pdgf38_propermab + pdgf38.csv
propermab_df_ginkgo: 246 rows <- GINKGO_propermab + ginkgo.csv
propermab_df_hutchinson2023enhancement_top200tm1_igg: 192 rows <- hutchinson2023enhancement_propermab + hutchinson2023enhancement_top200tm1_igg.csv
propermab_df_jetha2019homology_RT: 97 rows <- jetha2019homology_propermab + jetha2019homology_RT.csv


# Build models

Set **`MODEL_DATASET_KEYS`** here; this defines both **`datasets_standard`** and **`datasets_propermab`** from data you already loaded.


In [37]:
# Choose cohort here after: (1) main experimental+descriptor merges, (2) *Load Propermab features* (`propermab_dfs`).
# Change `MODEL_DATASET_KEYS` anytime — rerun **only this cell**, then selection CV, then fit. No Propermab reload needed.

MODEL_DATASET_KEYS = (
    "kraft2019herapin_relrt",
    "jain2024assessment",
    "jain2023identifying",
    "jain2017biophysical",
    "garbinski2023_tm1",
    "ab21",
    "pdgf38",
    "ginkgo",
    "hutchinson2023enhancement_top200tm1_igg",
    "jetha2019homology_RT",
)
# Examples:  ("pdgf38",)  or  ("ab21", "pdgf38")

if "propermab_dfs" not in globals():
    raise NameError(
        "Run the *Load Propermab features* cell first (defines propermab_dfs). "
        "Then set MODEL_DATASET_KEYS here."
    )

STANDARD_DF_BY_KEY = {
    "kraft2019herapin_relrt": df_kraft2019herapin_relrt,
    "jain2024assessment": df_jain2024assessment,
    "jain2023identifying": df_jain2023identifying,
    "jain2017biophysical": df_jain2017biophysical,
    "garbinski2023_tm1": df_garbinski2023_tm1,
    "ab21": df_ab21,
    "pdgf38": df_pdgf38,
    "ginkgo": df_ginkgo,
    "hutchinson2023enhancement_top200tm1_igg": df_hutchinson2023enhancement_top200tm1_igg,
    "jetha2019homology_RT": df_jetha2019homology_RT,
}

_missing_std = [k for k in MODEL_DATASET_KEYS if k not in STANDARD_DF_BY_KEY]
if _missing_std:
    raise KeyError(f"MODEL_DATASET_KEYS entries not in STANDARD_DF_BY_KEY: {_missing_std!r}")

_missing_prm = [k for k in MODEL_DATASET_KEYS if k not in propermab_dfs]
if _missing_prm:
    raise KeyError(
        f"No Propermab table for {_missing_prm!r}. "
        "Add the export under `*_propermab/features.csv` or remove those keys from MODEL_DATASET_KEYS."
    )

datasets_standard = [STANDARD_DF_BY_KEY[k] for k in MODEL_DATASET_KEYS]
datasets_propermab = [propermab_dfs[k].dropna() for k in MODEL_DATASET_KEYS]

datasets = list(datasets_standard)


In [38]:
# Outer CV feature selection: stability, RFE, forward SFS, and correlation (automl).
# All methods share the same folds per dataset. Results: `cv_selected_features_by_config[config_key]` -> list of dataset records.
# `stability_cv_by_selector` is an alias for the two stability_* entries only (backward compatible).
# On ElasticNet (etc.) non-convergence, try fallback estimators; if all fail, use a minimal feature set and continue.

from sklearn.exceptions import ConvergenceWarning

exclude_cols_stab = {
    "base",
    "heavy",
    "light",
    "name",
    "index",
}
low_variance_relative_std_threshold = 0.05
low_variance_epsilon = 1e-8
intercorr_threshold = 0.80
intercorr_importance_metric = "spearman"
INTERCORR_AFTER_SIG = 0.80
STABILITY_SELECTORS = ("elasticnet", "randomforest", "svm")
RFE_ESTIMATORS = ("elasticnet", "svm")
FORWARD_SFS_MODELS = ("elasticnet", "svm", "knn")
n_splits = 5
random_state = 42
stability_selection_verbose = False
SFS_INNER_CV = 5
SFS_N_FEATURE_FRAC = 0.15

# Catch non-convergence promoted to error inside sklearn/automl, and ordinary fit failures.
_CONVERGENCE_EXCEPTIONS = (Exception, ConvergenceWarning)


def _normalize_stab_estimator(name: str) -> str:
    n = str(name).strip().lower()
    allowed = {"elasticnet", "randomforest", "svm"}
    if n not in allowed:
        raise ValueError(f"stability selector must be one of {sorted(allowed)}; got {name!r}")
    return n


def _stability_model_fallbacks(primary: str) -> list[str]:
    primary = _normalize_stab_estimator(primary)
    order = ["elasticnet", "randomforest", "svm"]
    out: list[str] = []
    for x in [primary] + order:
        if x in order and x not in out:
            out.append(x)
    return out


def _rfe_estimator_fallbacks(primary: str) -> list[str]:
    primary = str(primary).strip().lower()
    order = ["elasticnet", "randomforest", "svm"]
    out: list[str] = []
    for x in [primary] + order:
        if x in order and x not in out:
            out.append(x)
    return out


def _sfs_model_fallbacks(primary: str) -> list[str]:
    primary = str(primary).strip().lower()
    order = ["elasticnet", "randomforest", "svm", "knn"]
    out: list[str] = []
    for x in [primary] + order:
        if x in order and x not in out:
            out.append(x)
    return out


def _collect_targets_out(
    mode: str,
    est,
    train_df_k,
    target_cols,
    kept_k,
    cur_by_t,
    random_state: int,
    config_key: str = "",
):
    union_feats = sorted({f for tc in target_cols for f in cur_by_t.get(tc, [])})
    if len(union_feats) == 0:
        union_feats = kept_k[:1]
    out = {}

    if mode == "stability":
        mt = _normalize_stab_estimator(est)
        for t in target_cols:
            freq, selected = None, None
            for try_mt in _stability_model_fallbacks(mt):
                try:
                    freq, selected = stability_selection_features(
                        train_df_k,
                        t,
                        union_feats,
                        n_subsamples=100,
                        sample_fraction=0.5,
                        model_type=try_mt,
                        l1_ratio=0.7,
                        alpha=0.01,
                        coef_threshold=0.05,
                        random_state=random_state,
                        verbose=stability_selection_verbose,
                    )
                    if try_mt != mt:
                        print(
                            f"{config_key}: stability primary {mt!r} did not converge for {t!r}; "
                            f"used {try_mt!r} instead."
                        )
                    break
                except _CONVERGENCE_EXCEPTIONS as e:
                    print(
                        f"{config_key}: stability {try_mt!r} target {t!r} failed "
                        f"({type(e).__name__}: {e}); trying next estimator…"
                    )
            if freq is None or selected is None:
                print(
                    f"{config_key}: stability all estimators failed for {t!r}; "
                    f"using empty selection for this target/split."
                )
                out[t] = {"selection_freq": None, "selected": []}
            else:
                out[t] = {"selection_freq": freq, "selected": selected}

    elif mode == "rfe":
        for t in target_cols:
            feats_t = list(cur_by_t.get(t, []))
            if len(feats_t) == 0:
                feats_t = kept_k[:1]
            n_pick = max(1, min(int(SFS_N_FEATURE_FRAC * len(train_df_k)), len(feats_t)))
            sel = None
            primary = str(est).strip().lower()
            for try_est in _rfe_estimator_fallbacks(primary):
                try:
                    sel = select_features_rfe(
                        train_df_k,
                        t,
                        n_pick,
                        candidate_features=feats_t,
                        rfe_estimator=try_est,
                        random_state=random_state,
                    )
                    if try_est != primary:
                        print(
                            f"{config_key}: RFE primary {primary!r} did not converge for {t!r}; "
                            f"used {try_est!r} instead."
                        )
                    break
                except _CONVERGENCE_EXCEPTIONS as e:
                    print(
                        f"{config_key}: RFE {try_est!r} target {t!r} failed "
                        f"({type(e).__name__}: {e}); trying next estimator…"
                    )
            if sel is None:
                print(
                    f"{config_key}: RFE all estimators failed for {t!r}; "
                    f"using fallback feature {feats_t[:1]!r}."
                )
                sel = feats_t[:1]
            out[t] = {"selection_freq": None, "selected": list(sel) if sel else feats_t[:1]}

    elif mode == "forward_sfs":
        for t in target_cols:
            feats_t = list(cur_by_t.get(t, []))
            if len(feats_t) == 0:
                feats_t = kept_k[:1]
            n_pick = max(1, int(SFS_N_FEATURE_FRAC * len(train_df_k)))
            n_pick = min(n_pick, len(feats_t))
            sel = None
            primary = str(est).strip().lower()
            for try_mt in _sfs_model_fallbacks(primary):
                try:
                    sel = select_features_forward_sfs(
                        split_train_df=train_df_k,
                        target_col=t,
                        candidate_features=feats_t,
                        n_features_to_select=n_pick,
                        model_type=try_mt,
                        cv=SFS_INNER_CV,
                        scoring="spearman",
                        random_state=random_state,
                        min_improvement=0.01,
                        n_jobs=-1,
                    )
                    if try_mt != primary:
                        print(
                            f"{config_key}: forward_sfs primary {primary!r} did not converge for {t!r}; "
                            f"used {try_mt!r} instead."
                        )
                    break
                except _CONVERGENCE_EXCEPTIONS as e:
                    print(
                        f"{config_key}: forward_sfs {try_mt!r} target {t!r} failed "
                        f"({type(e).__name__}: {e}); trying next model…"
                    )
            if sel is None:
                print(
                    f"{config_key}: forward_sfs all models failed for {t!r}; "
                    f"using fallback feature {feats_t[:1]!r}."
                )
                sel = feats_t[:1]
            out[t] = {"selection_freq": None, "selected": list(sel) if sel else feats_t[:1]}

    elif mode == "correlation":
        try:
            bundle = compute_correlation_bundle(
                train_df_k,
                target_cols,
                candidate_features=kept_k,
                exclude_cols=list(exclude_cols_stab),
                id_like_cols=(),
            )
        except _CONVERGENCE_EXCEPTIONS as e:
            print(f"{config_key}: correlation bundle failed ({type(e).__name__}: {e}); minimal features per target.")
            for t in target_cols:
                fb = cur_by_t.get(t, []) or kept_k[:1]
                out[t] = {"selection_freq": None, "selected": list(fb[:1]) if fb else list(kept_k[:1])}
            return out

        for t in target_cols:
            try:
                allowed = [f for f in cur_by_t.get(t, []) if f in train_df_k.columns]
                if len(allowed) == 0:
                    allowed = kept_k[:1]
                _, sig_tuples = select_features_by_target_correlation(
                    bundle,
                    t,
                    candidate_features=allowed,
                    p_threshold=0.05,
                    fdr_alpha=0.05,
                    use_fdr=True,
                )
                sig_features = [x[0] for x in sig_tuples]
                allowed_set = set(cur_by_t.get(t, []))
                sig_features = [f for f in sig_features if f in allowed_set]
                reduced = reduce_correlated_features(
                    split_train_df=train_df_k,
                    target_cols=target_cols,
                    feature_set={t: sig_features},
                    correlation_threshold=INTERCORR_AFTER_SIG,
                    importance_metric="spearman",
                )
                sel = list(reduced.get(t, []))
                if len(sel) == 0:
                    fb = cur_by_t.get(t, []) or kept_k[:1]
                    sel = list(fb[:1]) if fb else list(kept_k[:1])
                out[t] = {"selection_freq": None, "selected": sel}
            except _CONVERGENCE_EXCEPTIONS as e:
                print(
                    f"{config_key}: correlation target {t!r} failed ({type(e).__name__}: {e}); "
                    f"using minimal feature."
                )
                fb = cur_by_t.get(t, []) or kept_k[:1]
                out[t] = {"selection_freq": None, "selected": list(fb[:1]) if fb else list(kept_k[:1])}

    else:
        raise ValueError(f"unknown mode {mode!r}")

    return out


def _run_cv_for_mode(mode: str, est, config_key: str):
    cv_selected_features_by_config[config_key] = []
    print(f"\n========== {config_key} ==========")

    for d_i, df in enumerate(datasets):
        try:
            target_cols = [c for c in df.columns if str(c).startswith("target_")]
            if not target_cols:
                print(f"dataset index {d_i}: no target_* columns, skip")
                continue

            data = df.dropna(subset=target_cols).copy()
            N = len(data)
            if n_splits < 2 or n_splits > N:
                raise ValueError(f"dataset {d_i}: n_splits must be between 2 and N={N}.")

            base_size = N // n_splits
            remainder = N % n_splits
            segment_sizes = [base_size + (1 if i < remainder else 0) for i in range(n_splits)]
            boundaries = np.cumsum([0] + segment_sizes)

            rng = np.random.default_rng(random_state)
            idx = np.arange(N)
            rng.shuffle(idx)

            per_split_records = []

            for k in range(n_splits):
                test_start, test_end = boundaries[k], boundaries[k + 1]
                test_idx = idx[test_start:test_end]
                train_idx = np.concatenate([idx[:test_start], idx[test_end:]])
                train_df_k = data.iloc[train_idx].copy()

                candidate_features = [
                    c
                    for c in train_df_k.columns
                    if c not in exclude_cols_stab
                    and c not in target_cols
                    and not str(c).startswith("target")
                ]

                kept_k, _, rel_std_k = remove_low_variance_features(
                    X=train_df_k,
                    candidate_features=candidate_features,
                    relative_std_threshold=low_variance_relative_std_threshold,
                    epsilon=low_variance_epsilon,
                )
                if len(kept_k) == 0:
                    finite_feats = rel_std_k.index[np.isfinite(rel_std_k.values)].tolist()
                    kept_k = finite_feats if len(finite_feats) > 0 else candidate_features[:1]

                kept_k_set = set(kept_k)
                removed_k = [c for c in candidate_features if c not in kept_k_set]
                train_df_k = train_df_k.drop(columns=removed_k, errors="ignore").copy()

                prefilter_by_target_k = reduce_correlated_features(
                    split_train_df=train_df_k,
                    target_cols=target_cols,
                    feature_set=kept_k,
                    correlation_threshold=intercorr_threshold,
                    importance_metric=intercorr_importance_metric,
                )

                cur_by_t = {t: list(prefilter_by_target_k.get(t, [])) for t in target_cols}
                for t in target_cols:
                    if len(cur_by_t[t]) == 0:
                        cur_by_t[t] = kept_k[:1]

                targets_out = _collect_targets_out(
                    mode,
                    est,
                    train_df_k,
                    target_cols,
                    kept_k,
                    cur_by_t,
                    random_state,
                    config_key=config_key,
                )

                per_split_records.append(
                    {
                        "split_index": k,
                        "n_train": int(len(train_idx)),
                        "n_test": int(len(test_idx)),
                        "n_kept_lowvar": len(kept_k),
                        "targets": targets_out,
                    }
                )

            cv_selected_features_by_config[config_key].append(
                {
                    "dataset_index": d_i,
                    "config_key": config_key,
                    "selection_method": mode,
                    "selection_estimator": est,
                    "n_splits": n_splits,
                    "random_state": random_state,
                    "segment_sizes": segment_sizes,
                    "splits": per_split_records,
                }
            )
            print(
                f"{config_key} dataset {d_i}: CV done ({n_splits} splits, "
                f"test sizes {segment_sizes}, train sizes {[N - s for s in segment_sizes]})"
            )
        except _CONVERGENCE_EXCEPTIONS as e:
            print(
                f"{config_key} dataset {d_i}: skipped after error "
                f"({type(e).__name__}: {e}); continuing with next dataset / strategy."
            )
            continue



if "datasets_standard" not in globals() or "datasets_propermab" not in globals():
    raise NameError(
        "Define datasets_standard and datasets_propermab before this cell. "
        "Run: main merges → Load Propermab features (once) → Build models (sets both lists from MODEL_DATASET_KEYS)."
    )
if len(datasets_standard) != len(datasets_propermab):
    raise ValueError(
        f"datasets_standard ({len(datasets_standard)}) and datasets_propermab ({len(datasets_propermab)}) "
        "must have the same length and order (aligned MODEL_DATASET_KEYS)."
    )

cv_selected_features_by_source = {}

for _data_source_name, datasets in (
    ("standard", datasets_standard),
    ("propermab", datasets_propermab),
):
    print(f"\n{'#' * 18} {_data_source_name.upper()} — outer CV feature selection {'#' * 18}")
    cv_selected_features_by_config = {}

    for est in RFE_ESTIMATORS:
        _run_cv_for_mode("rfe", est, f"rfe_{est}")

    for est in FORWARD_SFS_MODELS:
        _run_cv_for_mode("forward_sfs", est, f"forward_sfs_{est}")

    for est in STABILITY_SELECTORS:
        mt = _normalize_stab_estimator(est)
        _run_cv_for_mode("stability", mt, f"stability_{mt}")

    _run_cv_for_mode("correlation", None, "correlation")

    cv_selected_features_by_source[_data_source_name] = cv_selected_features_by_config

cv_selected_features_by_config = cv_selected_features_by_source["standard"]
stability_cv_by_selector = {
    k.replace("stability_", "", 1): v
    for k, v in cv_selected_features_by_config.items()
    if k.startswith("stability_")
}

all_stability_features_reduced_across_datasets = stability_cv_by_selector[
    STABILITY_SELECTORS[-1]
]





################## STANDARD — outer CV feature selection ##################

========== rfe_elasticnet ==========
rfe_elasticnet dataset 0: CV done (5 splits, test sizes [26, 26, 26, 25, 25], train sizes [102, 102, 102, 103, 103])
rfe_elasticnet: RFE 'elasticnet' target 'target_ACSINS' failed (ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.397e+00, tolerance: 2.044e-01); trying next estimator…
rfe_elasticnet: RFE primary 'elasticnet' did not converge for 'target_ACSINS'; used 'randomforest' instead.
rfe_elasticnet: RFE 'elasticnet' target 'target_Tm' failed (ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.398e-01, tolerance: 7.884e-02); trying next estimator…
rfe_elasticnet: RFE primary 'elasticnet' did not con

KeyboardInterrupt: 

In [42]:
# Fit regressors on CV-selected features for EACH selector config and EACH max_feature_fraction.
# Runs separately for `datasets_standard` and `datasets_propermab`, then builds `winner_comparison_df`.

def _winner_features_fold_counts(w):
    """Map each regressor-input feature -> count of outer CV folds where it appears."""
    if not w:
        return {}
    fold_lists = w.get("cv_regressor_input_features") or []
    if not fold_lists:
        return {}
    from collections import Counter

    c = Counter()
    for fl in fold_lists:
        for name in set(fl):
            c[str(name)] += 1
    return dict(sorted(c.items(), key=lambda x: (-x[1], x[0])))


def _run_fit_benchmark_for_source(cv_selected_features_by_config, ds_fit):
    """Returns (selector_benchmark, winner_by_spearman, winner_by_pearson)."""
    if not cv_selected_features_by_config or all(
        len(v) == 0 for v in cv_selected_features_by_config.values()
    ):
        raise ValueError("cv_selected_features_by_config is empty for this data source.")

    exclude_cols_stab = {
        "base",
        "heavy",
        "light",
        "name",
        "index",
    }
    low_variance_relative_std_threshold = 0.05
    low_variance_epsilon = 1e-8

    enet_alpha = 0.01
    enet_l1_ratio = 0.7
    MAX_FEATURE_FRACTION_GRID = [0.15, 0.10, 0.08, 0.06, 0.04, 0.02]

    selector_benchmark = {}

    for config_key, stab_records in cv_selected_features_by_config.items():
        if not stab_records:
            continue
        selector_benchmark[config_key] = {}
        for max_feature_fraction in MAX_FEATURE_FRACTION_GRID:
            selector_benchmark[config_key][max_feature_fraction] = {}
            print(
                f"\n############ fit: config={config_key} max_feature_fraction={max_feature_fraction} ############"
            )

            for stab_record in stab_records:
                d_i = stab_record["dataset_index"]
                df = ds_fit[d_i]
                target_cols = [c for c in df.columns if str(c).startswith("target_")]
                if not target_cols:
                    continue

                data = df.dropna(subset=target_cols).copy()
                N = len(data)
                n_splits = stab_record["n_splits"]
                random_state = stab_record["random_state"]
                segment_sizes = list(stab_record["segment_sizes"])
                boundaries = np.cumsum([0] + segment_sizes)
                rng = np.random.default_rng(random_state)
                idx = np.arange(N)
                rng.shuffle(idx)

                bench_mf = selector_benchmark[config_key][max_feature_fraction]
                if d_i not in bench_mf:
                    bench_mf[d_i] = {}

                for target_col in target_cols:
                    results_per_split = []
                    cv_stability_selected = []
                    cv_regressor_input_features = []

                    for split_idx in range(n_splits):
                        if stab_record["splits"][split_idx]["split_index"] != split_idx:
                            raise RuntimeError(
                                f"Split order mismatch dataset {d_i} config {config_key}: "
                                f"expected {split_idx}, got {stab_record['splits'][split_idx]['split_index']}"
                            )

                        test_start, test_end = boundaries[split_idx], boundaries[split_idx + 1]
                        test_idx = idx[test_start:test_end]
                        train_idx = np.concatenate([idx[:test_start], idx[test_end:]])
                        train_df_k = data.iloc[train_idx].copy()
                        test_df_k = data.iloc[test_idx].copy()

                        candidate_features = [
                            c
                            for c in train_df_k.columns
                            if c not in exclude_cols_stab
                            and c not in target_cols
                            and not str(c).startswith("target")
                        ]
                        kept_k, _, rel_std_k = remove_low_variance_features(
                            X=train_df_k,
                            candidate_features=candidate_features,
                            relative_std_threshold=low_variance_relative_std_threshold,
                            epsilon=low_variance_epsilon,
                        )
                        if len(kept_k) == 0:
                            finite_feats = rel_std_k.index[np.isfinite(rel_std_k.values)].tolist()
                            kept_k = finite_feats if len(finite_feats) > 0 else candidate_features[:1]
                        kept_k_set = set(kept_k)
                        removed_k = [c for c in candidate_features if c not in kept_k_set]
                        train_df_k = train_df_k.drop(columns=removed_k, errors="ignore").copy()
                        test_df_k = test_df_k.drop(columns=removed_k, errors="ignore").copy()

                        tgt_entry = stab_record["splits"][split_idx]["targets"][target_col]
                        features = list(tgt_entry.get("selected") or [])
                        if len(features) == 0:
                            raise ValueError(
                                f"No selected features: config={config_key}, dataset={d_i}, "
                                f"target={target_col}, split={split_idx}."
                            )

                        selected_features, results_df, features_by_model = fit_and_compare_models(
                            merged_train=train_df_k,
                            merged_test=test_df_k,
                            target_col=target_col,
                            feature_list=features,
                            enet_alpha=enet_alpha,
                            enet_l1_ratio=enet_l1_ratio,
                            max_feature_fraction=max_feature_fraction,
                            make_plots=False,
                        )
                        cv_stability_selected.append(list(features))
                        cv_regressor_input_features.append(list(selected_features))
                        results_per_split.append(results_df)

                    results_agg = pd.concat(results_per_split, ignore_index=True)
                    summary = (
                        results_agg.groupby("model", as_index=False)
                        .agg(
                            mean_test_spearman=("spearman_r", "mean"),
                            mean_test_pearson=("pearson_r", "mean"),
                            mean_test_r2=("r2", "mean"),
                        )
                    )
                    best_idx = int(summary["mean_test_spearman"].idxmax())
                    best_row = summary.iloc[best_idx]
                    best_name = str(best_row["model"])

                    bench_mf[d_i][target_col] = {
                        "selection_config": config_key,
                        "max_feature_fraction": float(max_feature_fraction),
                        "best_regressor": best_name,
                        "mean_test_spearman": float(best_row["mean_test_spearman"]),
                        "mean_test_pearson": float(best_row["mean_test_pearson"]),
                        "mean_test_r2": float(best_row["mean_test_r2"]),
                        "cv_stability_selected": cv_stability_selected,
                        "cv_regressor_input_features": cv_regressor_input_features,
                        "per_regressor_cv_summary": summary.to_dict("records"),
                    }
                    print(
                        f"  [{config_key} mf={max_feature_fraction}] dataset {d_i} {target_col}: best={best_name} "
                        f"rho={best_row['mean_test_spearman']:.4f} r={best_row['mean_test_pearson']:.4f}"
                    )

    winner_by_spearman = {}
    winner_by_pearson = {}

    all_pairs = set()
    for sel_cfg, by_mf in selector_benchmark.items():
        for max_feature_fraction, by_ds in by_mf.items():
            for d_i, by_tgt in by_ds.items():
                for target_col in by_tgt.keys():
                    all_pairs.add((d_i, target_col))

    for d_i, target_col in sorted(all_pairs):
        cand = []
        for sel_cfg, by_mf in selector_benchmark.items():
            for max_feature_fraction, by_ds in by_mf.items():
                if d_i not in by_ds or target_col not in by_ds[d_i]:
                    continue
                entry = by_ds[d_i][target_col]
                summ = entry["per_regressor_cv_summary"]
                if not summ:
                    continue
                row_s = max(summ, key=lambda r: r["mean_test_spearman"])
                row_p = max(summ, key=lambda r: r["mean_test_pearson"])
                cand.append((sel_cfg, max_feature_fraction, entry, row_s, row_p))
        if not cand:
            continue

        best_s = max(cand, key=lambda x: x[3]["mean_test_spearman"])
        sel_s, mf_s, ent_s, rs, _ = best_s
        winner_by_spearman.setdefault(d_i, {})[target_col] = {
            "best_selection_config": sel_s,
            "max_feature_fraction": float(mf_s),
            "best_regressor": rs["model"],
            "mean_test_spearman": float(rs["mean_test_spearman"]),
            "mean_test_pearson": float(rs["mean_test_pearson"]),
            "mean_test_r2": float(rs["mean_test_r2"]),
            "cv_stability_selected": ent_s["cv_stability_selected"],
            "cv_regressor_input_features": ent_s["cv_regressor_input_features"],
        }

        best_p = max(cand, key=lambda x: x[4]["mean_test_pearson"])
        sel_p, mf_p, ent_p, _, rp = best_p
        winner_by_pearson.setdefault(d_i, {})[target_col] = {
            "best_selection_config": sel_p,
            "max_feature_fraction": float(mf_p),
            "best_regressor": rp["model"],
            "mean_test_spearman": float(rp["mean_test_spearman"]),
            "mean_test_pearson": float(rp["mean_test_pearson"]),
            "mean_test_r2": float(rp["mean_test_r2"]),
            "cv_stability_selected": ent_p["cv_stability_selected"],
            "cv_regressor_input_features": ent_p["cv_regressor_input_features"],
        }

    return selector_benchmark, winner_by_spearman, winner_by_pearson


if "cv_selected_features_by_source" not in globals():
    raise NameError(
        "Run the feature-selection CV cell first (expects cv_selected_features_by_source from standard + propermab)."
    )

selector_benchmark_by_source = {}
winner_by_spearman_by_source = {}
winner_by_pearson_by_source = {}

for _src_label in ("standard", "propermab"):
    print(f"\n{'#' * 20} FIT + WINNERS: {_src_label.upper()} {'#' * 20}")
    sb, ws, wp = _run_fit_benchmark_for_source(
        cv_selected_features_by_source[_src_label],
        datasets_standard if _src_label == "standard" else datasets_propermab,
    )
    selector_benchmark_by_source[_src_label] = sb
    winner_by_spearman_by_source[_src_label] = ws
    winner_by_pearson_by_source[_src_label] = wp

# Backward-compatible names (standard = original merged-descriptor pipeline)
selector_benchmark = selector_benchmark_by_source["standard"]
winner_by_spearman = winner_by_spearman_by_source["standard"]
winner_by_pearson = winner_by_pearson_by_source["standard"]
winner_by_dataset_target = winner_by_spearman

for _lbl, _ws, _wp in (
    ("STANDARD", winner_by_spearman_by_source["standard"], winner_by_pearson_by_source["standard"]),
    ("PROPERMAB", winner_by_spearman_by_source["propermab"], winner_by_pearson_by_source["propermab"]),
):
    print(f"\n=== {_lbl}: winner by mean test Spearman ===")
    for d_i in sorted(_ws.keys()):
        for t, w in _ws[d_i].items():
            print(
                f"  dataset {d_i} {t}: config={w['best_selection_config']} "
                f"max_frac={w['max_feature_fraction']} regressor={w['best_regressor']} "
                f"rho={w['mean_test_spearman']:.4f} r={w['mean_test_pearson']:.4f} "
                f"| stab_feats/split={[len(x) for x in w['cv_stability_selected']]} "
                f"reg_in/split={[len(x) for x in w['cv_regressor_input_features']]}"
            )
    print(f"\n=== {_lbl}: winner by mean test Pearson ===")
    for d_i in sorted(_wp.keys()):
        for t, w in _wp[d_i].items():
            print(
                f"  dataset {d_i} {t}: config={w['best_selection_config']} "
                f"max_frac={w['max_feature_fraction']} regressor={w['best_regressor']} "
                f"r={w['mean_test_pearson']:.4f} rho={w['mean_test_spearman']:.4f} "
                f"| stab_feats/split={[len(x) for x in w['cv_stability_selected']]} "
                f"reg_in/split={[len(x) for x in w['cv_regressor_input_features']]}"
            )


# Column guide for winner_comparison_df:
#   cohort_i / cohort / target — which study and endpoint.
#   std_* = merged FASTAb-style descriptors; prm_* = Propermab descriptor exports.
#   PearsonBest = chosen to maximize mean test Pearson r over (featsel × max_frac × regressor).
#   SpearmanBest = chosen to maximize mean test Spearman rho (same search).
#   mean_r / mean_rho = mean test Pearson / Spearman across outer CV folds for that winning regressor.
#   *_inputs = dict {feature_name: n_folds} — how many outer CV folds included that feature in regressor inputs.


def _winner_cols(w, block: str) -> dict:
    if w is None:
        return {
            f"{block}_featsel": pd.NA,
            f"{block}_model": pd.NA,
            f"{block}_inputs": {},
            f"{block}_max_frac": pd.NA,
            f"{block}_mean_r": pd.NA,
            f"{block}_mean_rho": pd.NA,
        }
    return {
        f"{block}_featsel": w["best_selection_config"],
        f"{block}_model": w["best_regressor"],
        f"{block}_inputs": _winner_features_fold_counts(w),
        f"{block}_max_frac": w["max_feature_fraction"],
        f"{block}_mean_r": float(w["mean_test_pearson"]),
        f"{block}_mean_rho": float(w["mean_test_spearman"]),
    }


if "MODEL_DATASET_KEYS" not in globals():
    raise NameError(
        "Define MODEL_DATASET_KEYS in the Build models cell (aligned with datasets_standard / datasets_propermab)."
    )

rows = []
n_ds = len(datasets_standard)
for d_i in range(n_ds):
    ds_name = MODEL_DATASET_KEYS[d_i] if d_i < len(MODEL_DATASET_KEYS) else d_i
    tset = set()
    for _lbl in ("standard", "propermab"):
        tset |= set(winner_by_spearman_by_source[_lbl].get(d_i, {}).keys())
        tset |= set(winner_by_pearson_by_source[_lbl].get(d_i, {}).keys())
    for tgt in sorted(tset):
        row = {
            "cohort_i": d_i,
            "cohort": ds_name,
            "target": tgt,
        }
        wp_std = winner_by_pearson_by_source["standard"].get(d_i, {}).get(tgt)
        wp_prm = winner_by_pearson_by_source["propermab"].get(d_i, {}).get(tgt)
        ws_std = winner_by_spearman_by_source["standard"].get(d_i, {}).get(tgt)
        ws_prm = winner_by_spearman_by_source["propermab"].get(d_i, {}).get(tgt)
        row.update(_winner_cols(wp_std, "std_PearsonBest"))
        row.update(_winner_cols(wp_prm, "prm_PearsonBest"))
        row.update(_winner_cols(ws_std, "std_SpearmanBest"))
        row.update(_winner_cols(ws_prm, "prm_SpearmanBest"))
        rows.append(row)

winner_comparison_df = pd.DataFrame(rows)
col_order = [
    "cohort_i",
    "cohort",
    "target",
    "std_PearsonBest_featsel",
    "std_PearsonBest_model",
    "std_PearsonBest_inputs",
    "std_PearsonBest_max_frac",
    "std_PearsonBest_mean_r",
    "std_PearsonBest_mean_rho",
    "prm_PearsonBest_featsel",
    "prm_PearsonBest_model",
    "prm_PearsonBest_inputs",
    "prm_PearsonBest_max_frac",
    "prm_PearsonBest_mean_r",
    "prm_PearsonBest_mean_rho",
    "std_SpearmanBest_featsel",
    "std_SpearmanBest_model",
    "std_SpearmanBest_inputs",
    "std_SpearmanBest_max_frac",
    "std_SpearmanBest_mean_r",
    "std_SpearmanBest_mean_rho",
    "prm_SpearmanBest_featsel",
    "prm_SpearmanBest_model",
    "prm_SpearmanBest_inputs",
    "prm_SpearmanBest_max_frac",
    "prm_SpearmanBest_mean_r",
    "prm_SpearmanBest_mean_rho",
]
winner_comparison_df = winner_comparison_df.reindex(columns=col_order)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 48)
print("\n=== winner_comparison_df (see column guide above) ===")
print(winner_comparison_df.to_string(index=False))
winner_comparison_df



#################### FIT + WINNERS: STANDARD ####################


KeyError: 'standard'

In [41]:
# Standard vs Propermab: when merged FASTAb descriptors underperform, print Propermab's winning {feature: n_folds} dicts.

from pprint import pprint

if "winner_comparison_df" not in globals() or winner_comparison_df.empty:
    print("Run the fit cell first (winner_comparison_df missing or empty).")
else:

    def _as_float(x):
        if x is None:
            return None
        try:
            if pd.isna(x):
                return None
        except TypeError:
            return None
        try:
            return float(x)
        except (TypeError, ValueError):
            return None

    print("=" * 80)
    print("Spearman: std_SpearmanBest_mean_rho < prm_SpearmanBest_mean_rho  →  Propermab inputs")
    print("=" * 80)
    n_s = 0
    for _, row in winner_comparison_df.iterrows():
        std_rho = _as_float(row.get("std_SpearmanBest_mean_rho"))
        prm_rho = _as_float(row.get("prm_SpearmanBest_mean_rho"))
        if std_rho is None or prm_rho is None:
            continue
        if std_rho < prm_rho:
            n_s += 1
            print(
                f"\n[{row['cohort']} | {row['target']}]  std_rho={std_rho:.4f}  prm_rho={prm_rho:.4f}"
            )
            print(f"  prm featsel={row.get('prm_SpearmanBest_featsel')}  model={row.get('prm_SpearmanBest_model')}")
            print("  prm_SpearmanBest_inputs (feature -> fold count):")
            pprint(row.get("prm_SpearmanBest_inputs"), width=120)
    if n_s == 0:
        print("(no cohort/target where standard Spearman < Propermab)")

    print("\n" + "=" * 80)
    print("Pearson: std_PearsonBest_mean_r < prm_PearsonBest_mean_r  →  Propermab inputs")
    print("=" * 80)
    n_p = 0
    for _, row in winner_comparison_df.iterrows():
        std_r = _as_float(row.get("std_PearsonBest_mean_r"))
        prm_r = _as_float(row.get("prm_PearsonBest_mean_r"))
        if std_r is None or prm_r is None:
            continue
        if std_r < prm_r:
            n_p += 1
            print(
                f"\n[{row['cohort']} | {row['target']}]  std_r={std_r:.4f}  prm_r={prm_r:.4f}"
            )
            print(f"  prm featsel={row.get('prm_PearsonBest_featsel')}  model={row.get('prm_PearsonBest_model')}")
            print("  prm_PearsonBest_inputs (feature -> fold count):")
            pprint(row.get("prm_PearsonBest_inputs"), width=120)
    if n_p == 0:
        print("(no cohort/target where standard Pearson < Propermab)")


Spearman: std_SpearmanBest_mean_rho < prm_SpearmanBest_mean_rho  →  Propermab inputs

[ab21 | target_viscosity]  std_rho=0.4800  prm_rho=0.7165
  prm featsel=rfe_elasticnet  model=ElasticNet
  prm_SpearmanBest_inputs (feature -> fold count):
{'neg_patch_area': 1, 'neg_ripley_k': 3, 'net_charge': 1}

Pearson: std_PearsonBest_mean_r < prm_PearsonBest_mean_r  →  Propermab inputs

[ab21 | target_viscosity]  std_r=0.6499  prm_r=0.9060
  prm featsel=stability_randomforest  model=SVR
  prm_PearsonBest_inputs (feature -> fold count):
{'neg_patch_area': 4, 'net_charge': 1}


In [77]:
len(df_ginkgo.columns) - 15

141

In [26]:
df_ginkgo['target_SEC_Monomer'].notna().sum()

242

In [44]:
# Drop rows where ANY target is missing (strict mode)
# Feature-selection pipeline examples:
#   "stability"
#   "correlation"
#   "forward_sfs"
#   "floating_sfs"
#   "rfe"
#   "stability->correlation"
#   "correlation->forward_sfs"
feature_selection_pipeline = "rfe"
# Base learner for stability, SFS (forward/floating), and RFE (single knob).
# Must be one of: elasticnet, randomforest, svm (linear SVR for svm).
model_type = "elasticnet"

# Forward SFS params
sfs_inner_cv = 5
sfs_scoring = "spearman"

low_variance_relative_std_threshold = 0.05
low_variance_epsilon = 1e-8

# Global inter-feature de-correlation prefilter (train-only, before pipeline)
intercorr_threshold = 0.80
intercorr_importance_metric = "spearman" # to target

exclude_cols = {
    "base",
    "heavy",
    "light",
    "name",
    "index",
}



pipeline_steps = [s.strip() for s in feature_selection_pipeline.split("->") if s.strip()]
valid_steps = {"stability", "correlation", "forward_sfs", "floating_sfs", "rfe"}
if len(pipeline_steps) == 0:
    raise ValueError("feature_selection_pipeline must contain at least one step.")
invalid = [s for s in pipeline_steps if s not in valid_steps]
if invalid:
    raise ValueError(
        f"Invalid pipeline steps: {invalid}. Allowed steps: {sorted(valid_steps)}"
    )


def _normalize_selection_estimator(name: str) -> str:
    n = str(name).strip().lower()
    allowed = {"elasticnet", "randomforest", "svm"}
    if n not in allowed:
        raise ValueError(
            f"Unknown selection_estimator {name!r}. Use one of: {sorted(allowed)}"
        )
    return n


model_type = _normalize_selection_estimator(model_type)


for df in datasets:
    target_cols = [col for col in df.columns.values.tolist() if col.startswith("target_")]
    data = df.dropna(subset=target_cols).copy()
    sfs_n_features_to_select = 0.1 * df.shape[0]

    n_splits = 5
    random_state = 42

    N = len(data)
    if n_splits < 2 or n_splits > N:
        raise ValueError(f"n_splits must be between 2 and N={N}.")

    base_size = N // n_splits
    remainder = N % n_splits
    segment_sizes = [base_size + (1 if i < remainder else 0) for i in range(n_splits)]
    boundaries = np.cumsum([0] + segment_sizes)

    rng = np.random.default_rng(random_state)
    idx = np.arange(N)
    rng.shuffle(idx)

    train_splits = []
    test_splits = []

    for i in range(n_splits):
        test_start, test_end = boundaries[i], boundaries[i + 1]
        test_idx = idx[test_start:test_end]
        train_idx = np.concatenate([idx[:test_start], idx[test_end:]])

        train_df_k = data.iloc[train_idx].copy()
        test_df_k = data.iloc[test_idx].copy()

        train_splits.append([(train_df_k.copy(), t) for t in target_cols])
        test_splits.append([(test_df_k.copy(), t) for t in target_cols])

    features_lowvar = []
    features_intercorr_prefilter_per_split = []
    features_intercorr_prefilter_union_counts = []
    selected_features_per_split = []

    # Optional debug/tracking holders for individual steps
    features_forward_sfs_per_split = []
    features_floating_sfs_per_split = []
    features_rfe_per_split = []
    features_stability_per_split = []
    features_significant_corrs_reduced_per_split = []

    for k in range(n_splits):
        train_df_k, _ = train_splits[k][0]
        test_df_k, _ = test_splits[k][0]

        candidate_features = [
            c
            for c in train_df_k.columns
            if c not in exclude_cols and c not in target_cols and not str(c).startswith("target")
        ]

        kept_k, removed_k, rel_std_k = remove_low_variance_features(
            X=train_df_k,
            candidate_features=candidate_features,
            relative_std_threshold=low_variance_relative_std_threshold,
            epsilon=low_variance_epsilon,
        )

        if len(kept_k) == 0:
            finite_feats = rel_std_k.index[np.isfinite(rel_std_k.values)].tolist()
            kept_k = finite_feats if len(finite_feats) > 0 else candidate_features[:1]

        kept_k_set = set(kept_k)
        removed_k = [c for c in candidate_features if c not in kept_k_set]

        train_df_k = train_df_k.drop(columns=removed_k, errors="ignore").copy()
        test_df_k = test_df_k.drop(columns=removed_k, errors="ignore").copy()

        # Train-only prefilter before pipeline starts.
        prefilter_by_target_k = reduce_correlated_features(
            split_train_df=train_df_k,
            target_cols=target_cols,
            feature_set=kept_k,
            correlation_threshold=intercorr_threshold,
            importance_metric=intercorr_importance_metric,
        )

        current_features_by_target = {
            t: list(prefilter_by_target_k.get(t, []))
            for t in target_cols
        }

        # Ensure non-empty starting set per target when possible.
        for t in target_cols:
            if len(current_features_by_target[t]) == 0:
                current_features_by_target[t] = kept_k[:1]

        # Sequential feature-selection pipeline.
        for step in pipeline_steps:
            if step == "stability":
                union_feats = sorted({f for t in target_cols for f in current_features_by_target.get(t, [])})
                if len(union_feats) == 0:
                    union_feats = kept_k[:1]

                selected_k = {}
                for t in target_cols:
                    try:
                        _, sel = stability_selection_features(
                            train_df_k,
                            t,
                            union_feats,
                            n_subsamples=100,
                            sample_fraction=0.5,
                            model_type=model_type,
                            l1_ratio=0.7,
                            alpha=0.01,
                            coef_threshold=0.05,
                            random_state=42,
                            verbose=True,
                        )
                        selected_k[t] = sel
                    except ValueError as e:
                        print(
                            f"Stability selection cannot be run for target={t!r}: {e}"
                        )
                        selected_k[t] = []

                next_features = {}
                for t in target_cols:
                    vals = list(selected_k.get(t, []))
                    next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
                current_features_by_target = next_features
                features_stability_per_split.append(current_features_by_target)

            elif step == "correlation":
                corr_df_k, significant_k = calculate_correlations_and_plot(
                    merged_df=train_df_k,
                    target_col=target_cols,
                    p_threshold=0.05,
                    fdr_alpha=0.05,
                    use_fdr=True,
                    normalize=False,
                    make_plots=False,
                    correlation_threshold=0,
                )

                sig_features_for_split = {}
                for t in target_cols:
                    sig_t = [el[0] for el in significant_k.get(t, [])]
                    allowed_t = set(current_features_by_target.get(t, []))
                    sig_features_for_split[t] = [f for f in sig_t if f in allowed_t]

                reduced_corr = reduce_correlated_features(
                    split_train_df=train_df_k,
                    target_cols=target_cols,
                    feature_set=sig_features_for_split,
                    correlation_threshold=0.8,
                    importance_metric="spearman",
                )

                next_features = {}
                for t in target_cols:
                    vals = list(reduced_corr.get(t, []))
                    next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
                current_features_by_target = next_features
                features_significant_corrs_reduced_per_split.append(current_features_by_target)

            elif step == "forward_sfs":
                selected_sfs_k = select_features_forward_sfs(
                    split_train_df=train_df_k,
                    target_cols=target_cols,
                    candidate_features=current_features_by_target,
                    n_features_to_select=sfs_n_features_to_select,
                    model_type=model_type,
                    cv=sfs_inner_cv,
                    scoring=sfs_scoring,
                    random_state=random_state,
                    min_improvement=0.01,
                    n_jobs=-1,
                )

                next_features = {}
                for t in target_cols:
                    vals = list(selected_sfs_k.get(t, []))
                    next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
                current_features_by_target = next_features
                features_forward_sfs_per_split.append(current_features_by_target)

            elif step == "floating_sfs":
                selected_sfs_k = select_features_floating_sfs(
                    split_train_df=train_df_k,
                    target_cols=target_cols,
                    candidate_features=current_features_by_target,
                    n_features_to_select=sfs_n_features_to_select,
                    model_type=model_type,
                    cv=sfs_inner_cv,
                    scoring=sfs_scoring,
                    random_state=random_state,
                    min_improvement=0.01,
                    n_jobs=-1,
                )

                next_features = {}
                for t in target_cols:
                    vals = list(selected_sfs_k.get(t, []))
                    next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
                current_features_by_target = next_features
                features_floating_sfs_per_split.append(current_features_by_target)

            elif step == "rfe":
                from sklearn.feature_selection import RFE
                from sklearn.svm import SVR

                selected_rfe_k = {}
                for t in target_cols:
                    feats_t = list(current_features_by_target.get(t, []))
                    if len(feats_t) == 0:
                        selected_rfe_k[t] = []
                        continue

                    cols_t = feats_t + [t]
                    work_t = train_df_k[cols_t].replace([np.inf, -np.inf], np.nan).dropna()
                    if len(work_t) < 3:
                        selected_rfe_k[t] = feats_t[:1]
                        continue

                    X_t = work_t[feats_t].apply(pd.to_numeric, errors="coerce")
                    y_t = pd.to_numeric(work_t[t], errors="coerce")
                    mask_t = y_t.notna() & X_t.notna().all(axis=1)
                    X_t = X_t.loc[mask_t]
                    y_t = y_t.loc[mask_t]

                    if len(X_t) < 3:
                        selected_rfe_k[t] = feats_t[:1]
                        continue

                    n_select_t = max(1, min(int(sfs_n_features_to_select), X_t.shape[1]))
                    if model_type == "elasticnet":
                        estimator = ElasticNet(alpha=0.02, l1_ratio=0.7, max_iter=5000, random_state=random_state)
                    elif model_type == "randomforest":
                        estimator = RandomForestRegressor(n_estimators=100, random_state=random_state)
                    elif model_type == "svm":
                        estimator = SVR(kernel="linear", C=1.0, epsilon=0.1)
                    else:
                        raise ValueError(
                            f"Unknown model_type {model_type!r} (expected elasticnet, randomforest, svm)."
                        )
                    selector = RFE(estimator=estimator, n_features_to_select=n_select_t, step=1)
                    selector.fit(X_t, y_t)
                    selected_cols_t = X_t.columns[selector.support_].tolist()
                    selected_rfe_k[t] = selected_cols_t if len(selected_cols_t) > 0 else feats_t[:1]

                next_features = {}
                for t in target_cols:
                    vals = list(selected_rfe_k.get(t, []))
                    next_features[t] = vals if len(vals) > 0 else list(current_features_by_target.get(t, []))
                current_features_by_target = next_features
                features_rfe_per_split.append(current_features_by_target)

        train_splits[k] = [(train_df_k.copy(), t) for t in target_cols]
        test_splits[k] = [(test_df_k.copy(), t) for t in target_cols]

        features_lowvar.append(kept_k)
        features_intercorr_prefilter_per_split.append(prefilter_by_target_k)
        features_intercorr_prefilter_union_counts.append(
            len(sorted({f for t in target_cols for f in prefilter_by_target_k.get(t, [])}))
        )

        selected_features_per_split.append(current_features_by_target)

    print("Feature-selection pipeline:", " -> ".join(pipeline_steps))
    print("Low-variance kept feature counts per split:", [len(x) for x in features_lowvar])
    print("Inter-corr prefilter kept feature counts per split:", features_intercorr_prefilter_union_counts)
    if features_forward_sfs_per_split:
        print(
            "Forward SFS kept feature counts per split:",
            {t: [len(x[t]) for x in features_forward_sfs_per_split] for t in target_cols},
        )
    if features_floating_sfs_per_split:
        print(
            "Floating SFS kept feature counts per split:",
            {t: [len(x[t]) for x in features_floating_sfs_per_split] for t in target_cols},
        )
    if features_rfe_per_split:
        print(
            "RFE kept feature counts per split:",
            {t: [len(x[t]) for x in features_rfe_per_split] for t in target_cols},
        )
    print(
        f"Created {n_splits} splits: "
        f"test sizes {segment_sizes}, train sizes {[N - s for s in segment_sizes]}."
    )




Feature-selection pipeline: rfe
Low-variance kept feature counts per split: [122, 123, 121, 121, 121]
Inter-corr prefilter kept feature counts per split: [116, 115, 114, 115, 114]
RFE kept feature counts per split: {'target_SEC_Monomer': [24, 24, 24, 24, 24], 'target_SMAC': [24, 24, 24, 24, 24], 'target_HIC': [24, 24, 24, 24, 24], 'target_HAC': [24, 24, 24, 24, 24], 'target_PR_CHO': [24, 24, 24, 24, 24], 'target_PR_Ova': [24, 24, 24, 24, 24], 'target_AC_SINS_pH6_0': [24, 24, 24, 24, 24], 'target_AC_SINS_pH7_4': [24, 24, 24, 24, 24], 'target_Tonset': [24, 24, 24, 24, 24], 'target_Tm1': [24, 24, 24, 24, 24], 'target_Tm2': [24, 24, 24, 24, 24]}
Created 5 splits: test sizes [14, 14, 14, 14, 14], train sizes [56, 56, 56, 56, 56].


In [107]:
for target_col in target_cols:
    results_per_split = []

    for split_idx in range(n_splits):
        train_df, _ = train_splits[split_idx][0]
        test_df, _ = test_splits[split_idx][0]

        # selected_features_per_split stores dict[target_col] -> list[str]
        feature_map = selected_features_per_split[split_idx]
        features = feature_map.get(target_col, [])

        if len(features) == 0:
            raise ValueError(
                f"No selected features for target={target_col}, split={split_idx}. "
                "Check feature-selection outputs before model fitting."
            )

        enet_alpha = 0.01

        selected_features, results_df, features_by_model = fit_and_compare_models(
            merged_train=train_df,
            merged_test=test_df,
            target_col=target_col,
            feature_list=features,
            enet_alpha=enet_alpha,
            enet_l1_ratio=0.7,
            max_feature_fraction=0.1,
            make_plots=False,
        )
        print(f"ENet top-|coef| feature set (n={len(selected_features)}): {selected_features}")
        for mname, mfeats in sorted(features_by_model.items()):
            print(f"  {mname}: n_active={len(mfeats)} -> {mfeats}")

        results_per_split.append(results_df)

    results_agg = pd.concat(results_per_split, ignore_index=True)
    summary = results_agg.groupby("model").agg({"pearson_r": "mean", "spearman_r": "mean", "r2": "mean"}).reset_index()
    print("Mean test metrics across splits:")
    print(target_col)
    print(summary)

NameError: name 'train_splits' is not defined

In [ ]:
['cluster_metrics_neg_ann_index', 'salt_bridges_metrics_number_of_salt_bridges', 'charge_metrics_light_charge_pH74']
['cluster_metrics_positive_exposed_cluster_largest_size', 'salt_bridges_metrics_number_of_salt_bridges']
['cluster_metrics_positive_exposed_cluster_largest_size', 'h_bonds_metrics_avg_hbond_cdr', 'charge_metrics_dipole_moment_magnitude']
['charge_metrics_heavy_charge_pH74', 'salt_bridges_metrics_avg_salt_inter_chain', 'density_metrics_avg_negative_cdr_over_cdr']
['cluster_metrics_pnc_all_surface_exposed', 'salt_bridges_metrics_number_of_salt_bridges', 'h_bonds_metrics_avg_hbond_cdr']
Mean test metrics across splits:
target_viscosity
          model  pearson_r  spearman_r
0    ElasticNet   0.831759    0.674496
1        Linear   0.831529    0.664972
2  RandomForest   0.845018    0.622115
3         Ridge   0.831555    0.664972

# Propermab

### Propermab data

Merged Propermab tables live in **`propermab_dfs`** / **`propermab_df_*`** from the **Load Propermab features** cell (earlier in the notebook). To compare standard vs Propermab on a chosen cohort, edit **`MODEL_DATASET_KEYS`** in **Build models** and rerun that cell, then **selection CV**, then **fit** — no need to reload Propermab CSVs.


The **Build models** cell builds `datasets_standard` and `datasets_propermab` from `MODEL_DATASET_KEYS`. The **selection** and **fit** cells below mirror the *Build models* pipeline and fill `cv_selected_features_by_source`, `selector_benchmark_by_source`, and `winner_comparison_df`.


In [17]:
cfg, mf = "stability_elasticnet", 0.02
d_i = 0  # dataset index in `datasets`
target_col = "target_fitness"  # e.g. "target_Tm1"

entry = selector_benchmark[cfg][mf][d_i][target_col]

print("best_regressor (mean CV Spearman):", entry["best_regressor"])
print("SVR row in summary:", next(r for r in entry["per_regressor_cv_summary"] if r["model"] == "SVR"))

# Features actually fed to every regressor (incl. SVR) after EN |coef| cap — same list for all models each fold
for k, feats in enumerate(entry["cv_regressor_input_features"]):
    print(f"fold {k} (n={len(feats)}): {feats}")

# Optional: stability output before that cap
for k, feats in enumerate(entry["cv_stability_selected"]):
    print(f"fold {k} stability only (n={len(feats)}): {feats}")

best_regressor (mean CV Spearman): SVR
SVR row in summary: {'model': 'SVR', 'mean_test_spearman': 0.849085219397342, 'mean_test_pearson': 0.834576975139503, 'mean_test_r2': 0.6295687830054076}
fold 0 (n=2): ['net_charge', 'dipole_moment']
fold 1 (n=2): ['net_charge', 'dipole_moment']
fold 2 (n=2): ['net_charge', 'dipole_moment']
fold 3 (n=2): ['net_charge', 'dipole_moment']
fold 4 (n=2): ['net_charge', 'dipole_moment']
fold 0 stability only (n=4): ['net_charge', 'dipole_moment', 'pos_patch_area', 'scm']
fold 1 stability only (n=4): ['net_charge', 'pos_patch_area', 'dipole_moment', 'pos_ripley_k']
fold 2 stability only (n=6): ['net_charge', 'pos_patch_area', 'dipole_moment', 'net_charge_cdr', 'exposed_net_charge', 'hyd_patch_area_cdr']
fold 3 stability only (n=4): ['net_charge', 'dipole_moment', 'pos_patch_area', 'net_charge_cdr']
fold 4 stability only (n=6): ['net_charge', 'dipole_moment', 'pos_patch_area', 'exposed_net_charge_cdr', 'net_charge_cdr', 'exposed_net_charge']


In [37]:
for target_col in target_cols:
    results_per_split = []

    for split_idx in range(n_splits):
        train_df, _ = train_splits[split_idx][0]
        test_df, _ = test_splits[split_idx][0]

        # selected_features_per_split stores dict[target_col] -> list[str]
        feature_map = selected_features_per_split[split_idx]
        features = feature_map.get(target_col, [])

        if len(features) == 0:
            raise ValueError(
                f"No selected features for target={target_col}, split={split_idx}. "
                "Check feature-selection outputs before model fitting."
            )

        enet_alpha = 0.01

        selected_features, results_df, features_by_model = fit_and_compare_models(
            merged_train=train_df,
            merged_test=test_df,
            target_col=target_col,
            feature_list=features,
            enet_alpha=enet_alpha,
            enet_l1_ratio=0.7,
            max_feature_fraction=0.1,
            make_plots=False,
        )
        print(f"ENet top-|coef| feature set (n={len(selected_features)}): {selected_features}")
        for mname, mfeats in sorted(features_by_model.items()):
            print(f"  {mname}: n_active={len(mfeats)} -> {mfeats}")

        results_per_split.append(results_df)

    results_agg = pd.concat(results_per_split, ignore_index=True)
    summary = results_agg.groupby("model").agg({"pearson_r": "mean", "spearman_r": "mean", "r2": "mean"}).reset_index()
    print("Mean test metrics across splits:")
    print(target_col)
    print(summary)

['feature_hc_subtype']
['feature_hc_subtype']
['feature_hc_subtype', 'neg_ann_index', 'dipole_moment', 'exposed_net_charge_cdr', 'aromatic_cdr']
['feature_hc_subtype', 'neg_ann_index', 'exposed_net_charge_cdr', 'exposed_net_charge', 'aromatic_ripley_k']
['feature_hc_subtype']
Mean test metrics across splits:
target_SEC_Monomer
          model  pearson_r  spearman_r        r2
0    ElasticNet   0.568024    0.379994 -0.884299
1        Linear   0.567000    0.378236 -0.892208
2  RandomForest   0.594665    0.360222  0.212703
3         Ridge   0.567011    0.378236 -0.891960
4           SVR   0.485511    0.266413  0.185593
5           kNN   0.569679    0.398903  0.089259
['aromatic_asa', 'hyd_patch_area_cdr', 'cdr_h3_length']
['hyd_patch_area_cdr', 'cdr_h3_length', 'feature_hc_subtype']
['cdr_h3_length', 'aromatic_ripley_k']
['hyd_patch_area_cdr', 'cdr_h3_length', 'aromatic_asa', 'feature_hc_subtype']
['hyd_patch_area_cdr', 'neg_ripley_k', 'cdr_h3_length', 'aromatic_asa']
Mean test metrics acr

# Analyze correlations between propermab and our features

In [3]:
def _repo_root() -> Path:
    """Resolve repo root whether the notebook cwd is the repo or ``src/``."""
    here = Path.cwd().resolve()
    for base in (here, here.parent):
        if (base / "data" / "pdgf38.csv").is_file():
            return base
    raise FileNotFoundError("Could not locate data/pdgf38.csv; run the notebook from the repo root or from src/.")


REPO = _repo_root()
NAME_COL = "name"

exp_path = REPO / "data" / "pdgf38.csv"
propermab_path = REPO / "pdgf38_propermab" / "features.csv"
ours_dir = REPO / "pdgf38_results"

exp = pd.read_csv(exp_path)
propermab = pd.read_csv(propermab_path)
ours = load_json_results(ours_dir)

for df in (exp, propermab, ours):
    if NAME_COL not in df.columns:
        raise ValueError(f"Missing {NAME_COL!r} column in {df.shape}")

# Align key dtype for stable merges (handles int vs str IDs)
exp[NAME_COL] = exp[NAME_COL].astype(str)
propermab[NAME_COL] = propermab[NAME_COL].astype(str)
ours[NAME_COL] = ours[NAME_COL].astype(str)

merged_propermab = exp.merge(propermab, on=NAME_COL, how="inner", suffixes=("", "_propermab"))
merged_ours = exp.merge(ours, on=NAME_COL, how="inner", suffixes=("", "_ours"))

merged_propermab = merged_propermab.sort_values(NAME_COL).reset_index(drop=True)
merged_ours = merged_ours.sort_values(NAME_COL).reset_index(drop=True)

print(f"exp: {len(exp)} rows; Propermab merge: {len(merged_propermab)} rows; our JSON merge: {len(merged_ours)} rows")

ValueError: Missing 'name' column in (38, 28)

In [7]:
propermab['pdb_file'].iloc[0]

'/storage/antibody_data/PairedStructures/FASTAb/pdgf38/AB-001.pdb'